<img src="../../images/SnowparkIconLabel.png" alt="Snowpark Icon" width=150px align=right /> 


# Data Science Training Utilities
This notebook contains utility functions and objects used by other notebooks. <span style="color:red;font-weight:bold">None of the cells in this notebook were intended to be run in their own kernel.</span> This notebook is to be used by **other** notebooks like this:
```python
%run ../utils/ds_utils_python.ipynb
```

or
```python
%run ../../utils/ds_utils_python.ipynb
```


The `../` and `../../` above indicates a relative path from the referencing notebook to this notebook. The path might vary depending on the location of the calling notebook.

#### Functions
- [Create Key/Pair Authentication Objects](#kp)
    - `create_key_pair_auth_objects()`
- [Create a Session Object](#cs)
    -  `create_session()`
- [Confirm or Create Context Objects ](#create_ctx)
    - `confirm_or_create_lesson_context(Session,str)`
- [Close the Session and Clean up the Context Items](#clean)
    - `close_session_and_clean_up(Lesson)`
- [Utility Functions](#util_functions)
    - [Session Information](#session_info)
    - [OS Information](#os_info)
    - [Version Checking](#version_checking)
    - [Available Packages](#packages)
    
#### Classes
- [The Lesson Class](#lesson_class)
- [Lesson Creation](#lesson_creation)
    - [`confirm_or_create_lesson_context`](#confirm_or_create_lesson_context)

In [1]:
from snowflake.snowpark import functions, Session, DataFrame
from snowflake.snowpark.types import StringType

---

<a id="kp"></a><a id="props"></a><a id="bytes"></a>
### Create Key/Pair Authentication Objects

With each notebook, we need to read properties and private key information from `/home/jovyan/.ssh/sf_config`. These objects are created in the first exercise of the course and used by all subsequent lecture notebooks, appendix notebooks, and lab notebooks. 

The function `create_key_pair_auth_objects()` reads from the above file and sets global variables that are used when `create_session(...)` is invoked. These objects are returned:
- `props` - a dictionary of key/value pairs for the authentication mechanism
- `private_key_bytes` - a byte array of the user's private key

A user can create different sessions for different notebooks or multiple sessions in the same notebook. However, `create_session(...)` can't be used to create two session where the user is different unless the credentials are modified in between creating the sessions. 

The function `create_session(...)` calls `create_key_pair_auth_objects()`. It is not necessary to call `create_key_pair_auth_objects()` prior to calling `create_session(...)`. 

In [2]:
# Used by function create_session(...)
def create_key_pair_auth_objects() -> None:
    
    # Used when creating a session
    props = {}
    private_key_bytes = None
    
    # Path to the key/pair auth file
    configDir = "/home/jovyan/.ssh"
    configFile = configDir + "/sf_config"
    
    # Load configuration file
    with open(configFile) as f:
        lines = f.readlines()
        
    # Convert configuration to a properties map
    for line in lines:
        (key, value) = line.split('=')
        props.update({key.lower() : value[0:-1]})
        
    # Convert the private key to a DER-encoded bytes object
    from cryptography.hazmat.primitives import serialization
    from cryptography.hazmat.backends import default_backend
    
    with open(props['private_key_file'], "rb") as key:
        private_key = serialization.load_pem_private_key(
            key.read(),
            password=None,
            backend=default_backend()
        )
    private_key_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    print(f"Read of key/pair authentication objects successful")
    return (props, private_key_bytes)
# end create_key_pair_auth_objects() 

<a id="cs"></a>

---

### Create A Session Object

The function `create_session()` will fail without the objects `props` and `private_key_bytes` that are created using function `create_key_pair_auth_objects()`. 

In [3]:
from snowflake.snowpark import Session
def create_session() -> Session:
    try:
        (props,private_key_bytes) = create_key_pair_auth_objects()
    except Exception as ex:
        print(f"Could not create/read key/pair authentication objects: {str(ex)}")
        print(f"Is it possible that you have not already completed the key pair authentication exercise?")
        raise ex
    session = None
    try:
        session = (Session
                .builder      # props and private_key_bytes are created in function create_key_pair_auth_objects()
                .configs({**props, **{"private_key": private_key_bytes}})
                .create()
        )           
    except Exception as ex:
        print(f"Problem creating session: {str(ex)}")
        raise ex 
    print("User authenticated and session created")
    display_session_info(session)
    return session

---

<a id="create_ctx"></a>
### Confirm or Create Context Objects

With each lesson, we need to confirm that the `Session` has the following context items set:
- Role
- Warehouse
- Database
- Schema

The default context values used here might not be the defaults set on the user's `USER` object in the Snowflake account. Rather than hoping the defaults are set on the user, we set all context items here to be safe. 

Additionally, we want to set the `query_tag` for the `Session` to a value that includes the username and lesson name. This makes for easier access in the query history.

---

<a id="lesson_class"></a>
#### The Lesson Class

The function `confirm_or_create_lesson_context` takes as an argument a `Session` object and a lesson name as a string.

It returns a `Lesson` object that has reflective functions:
- account()
- user()
- role()
- warehouse()
- database()
- schema()
- full_schema()

Additionally, the `Lesson` object contains a `DataFrame` for the above context items that can be accessed with `lessonObj.contextDF`. 

In [4]:
from snowflake.snowpark import Session,DataFrame

# Class representing a lesson or lab
class Lesson:
    
    session_closed = False
    
    def __init__(self,lesson_name:str,sess:Session):
        if(sess == None):
            raise Exception("Session is None; Lesson cannot be created without a Session object.")
        self.lesson_name = lesson_name        
        self.session = sess
        self.contextDF = (
            session.create_dataframe([""])
                .select(
                 functions.lit(self.account()).alias("Account")
                ,functions.current_user().alias("User")
                ,functions.current_role().alias("Role")
                ,functions.current_warehouse().alias("Warehouse")
                ,functions.current_database().alias("Database")
                ,functions.current_schema().alias("Schema")
              )
        )
        
    def get_lesson_name(self) -> str:
        return self.lesson_name
    
    def lesson_name(self) -> str:
        return f"{self.get_lesson_name()}"
    
    def show_lesson_context(self) -> None:
        self.contextDF.show()
        
    def schema(self) -> str:
        return self.session.get_current_schema().strip('"')
    
    def get_schema_name(self) -> str:
        return f"{self.schema()}"
    
    def schema_name(self) -> str:
        return f"{self.schema()}"
    
    def full_schema(self) -> str:
        return f"{self.database()}.{self.schema()}"
    
    def fully_qualified_schema(self) -> str:
        return f"{self.full_schema()}"    
    
    def fully_qualified_schema_name(self) -> str:
        return f"{self.full_schema()}"
    
    def get_full_schema(self) -> str:
        return f"{self.full_schema()}"
        
    def database(self) -> str:
        return self.session.get_current_database().strip('"')
    
    def db(self) -> str:
        return f"{self.database()}"
    
    def db_name(self) -> str:
        return f"{self.database()}"
    
    def get_db_name(self) -> str:
        return f"{self.database()}"
    
    def get_database(self) -> str:
        return f"{self.database()}"
    
    def user(self) -> str:
        return self.contextDF.collect()[0].USER
    
    def username(self) -> str:
        return f"{self.user()}"
    
    def user_name(self) -> str:
        return f"{self.user()}"
    
    def get_username(self) -> str:
        return f"{self.user()}"
        
    def role(self) -> str:
        return self.session.get_current_role().strip('"')
        
    def account(self) -> str:
        return self.session.get_current_account().strip('"')
    
    def account_name(self) -> str:
        return f"{self.account()}"
    
    def get_account(self) -> str:
        return f"{self.account()}"
        
    def warehouse(self) -> str:
        return self.session.get_current_warehouse().strip('"')
    
    def get_session(self) -> Session:
        return self.session
    
    def is_session_closed(self) -> bool:
        return self.session_closed
    
    # Used by each notebook to close the current session
    def close_session(self) -> None:
        if(self.session_closed):
            return None
        self.session.close()
        self.session_closed = True
        self.session = None
        
# End class Lesson        

<a id="lesson_creation"></a>
#### Lesson Creation

<a id="confirm_or_create_lesson_context"></a>

In [ ]:
# Creates the context objects if they don't exist and sets session context items
def confirm_or_create_lesson_context(sess:Session, lesson_name:str, keep_schema=False) -> Lesson:        
    # Confirm a session
    if(sess is None):
        print("None sent as a Session argument. Can't create context items.")
        return
    
    print("Creation of context items could take a few moments... be patient")   
    
    # Programmatically retrieve the current username
    username = get_user_name_from_session(sess)
    print(f"The current user is: {username}")
    
    # Dynamically name our demo objects    
    db_name = f"{username}_DB"
    schema_name = f"{lesson_name}_LESSON"
    full_schema_name = f"{db_name}.{schema_name}"
    wh_name = f"{username}_WH"
    default_role = "ACCOUNTADMIN" # My hack
    
    # The status column will be used when confirming, creating, and setting the context items
    status_column = functions.col("\"status\"")
    
    # Set our current role appropriately
    sess.use_role(default_role) # default_role s/b TRAINING_ROLE
    
    # Set the query tag for operations in this lesson
    sess.query_tag = f"Data Science: {username} - {lesson_name}"
    
    # Verify query tag and print it
    print(f"Query tag is: {sess.query_tag}")
    
    # Ensure we have a warehouse set in our context with the proper configuration
    (sess.sql(f"CREATE WAREHOUSE IF NOT EXISTS {wh_name}")
            .rename(status_column, f"Created warehouse {wh_name}")
            .show(1,80)
    )
    (sess.sql(f"ALTER WAREHOUSE IF EXISTS {wh_name} SET AUTO_RESUME = true")
            .rename(status_column, f"Altered warehouse {wh_name}")
            .show(1,80)
    )
    print(f"Setting current warehouse to {wh_name}")
    sess.use_warehouse(wh_name)

    # Create (if not present) and confirm the database and schema
    print(f"Creating database {db_name}")
    (sess.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")
            .rename(status_column,f"Created database {db_name}")
            .show(1,80)
    )
    if(keep_schema == False):
        print(f"Creating schema {schema_name}")
        (sess.sql(f"CREATE OR REPLACE SCHEMA {full_schema_name}")
                .rename(status_column, f"Created schema {full_schema_name}")
                .show(1,80)
        )
    else:
        (sess.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema_name}")
                .rename(status_column, f"Created schema {full_schema_name}")
                .show(1,80)
        )
    
    # Set our current namespace appropriately
    print(f"Setting current namespace to {full_schema_name}")
    sess.use_schema(full_schema_name)   
    
    # Create Lesson object
    lesson = Lesson(lesson_name,sess)    
    
    print("Lesson object created as lesson")
    lesson.show_lesson_context()
    
    print("Use lesson.show_lesson_context() for context information")
    print("========== READY ===========")
    return lesson

def get_lesson() -> Lesson:
    try: lesson
    except NameError: return None
    else: return lesson

---

<a id="util_functions"></a>

### Utility Functions

<a id="session_info"></a>

#### Session Information

In [6]:
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session

def get_active_session() -> Session:
    return get_active_session()

def get_account_from_session(sess:Session) -> str:
    return sess.get_current_account().strip('"')

# Session has no function current_user(). But the functions object does. 
def get_username_from_session(sess:Session) -> str:
    from snowflake.snowpark import functions
    return str(sess.create_dataframe([""]).select(functions.current_user()).collect()[0][0])

def get_user_from_session(sess:Session) -> str:
    return get_username_from_session(sess)

def get_user_name_from_session(sess:Session) -> str:
    return get_username_from_session(sess)

def username_from_session(sess:Session) -> str:
    return get_username_from_session(sess)

def get_snowflake_session_ID(sess:Session) -> str:
    from snowflake.snowpark import functions
    return (str(sess.create_dataframe([""])
        .select(
            functions.current_session()
         )
        .collect()[0][0]
      )
    )

def get_python_connector_session_ID(sess:Session) -> str:
    return get_session_info_item(sess, "python.connector.session.id")

def get_snowpark_version_str(sess:Session) -> str:
    return get_session_info_item(sess, "version")

def snowpark_version() -> tuple:
    from snowflake.snowpark import VERSION
    return VERSION

def get_snowpark_version() -> tuple:
    return snowpark_version()

def snowpark_major_release_version() -> int:    
    return snowpark_version()[0]

def get_snowpark_major_release_version() -> int:
    return snowpark_major_release_version()

def snowpark_minor_release_version() -> int:    
    return snowpark_version()[1] 

def get_snowpark_minor_release_version() -> int:
    return snowpark_minor_release_version()

def snowpark_maintenance_release_version() -> int:    
    return snowpark_version()[2] 

def get_maintenance_release_version() -> int:
    return snowpark_maintenance_release_version()

def get_session_info_item(sess:Session, key:str) -> str:
    itemsArray = get_session_info_DF(sess).collect()
    for row in itemsArray:
        k = str(row[0])
        v = str(row[1])
        if(k == key):
            return v
    return None

def get_session_info_DF(sess:Session) -> DataFrame:
    # Retrieve the session info as text
    sess_info = str(sess._session_info)
    
    # Column objects used to parse the text
    new_line_col = functions.lit("\n")     # A Column holding a literal new line character
    value_col = functions.col("VALUE")
    colon_col = functions.lit(":")        # A Column holding a literal colon
    trim_chars_col = functions.lit(" \",") # A Column holding the characters to trim from keys and values
    
    # A Column with trimmed session information
    session_info_col = (functions          
            .trim(
                 functions.lit(sess_info)
                ,new_line_col
             )
    )
    
    # A table function object representing the Snowflake function SPLIT_TO_TABLE
    split_to_table_func = functions.table_function("split_to_table")
    
    # Create the session info DataFrame
    session_info_DF = (sess
     .table_function(
       split_to_table_func(
            session_info_col # Column to split
           ,new_line_col     # Split on a new line
        )              
     )
    .select(
        
        # First column selected
         functions.trim(
            functions.split(
                value_col          # Line to split
               ,colon_col          # Split on a colon (:)
             )[0]                 # First token from split
               .cast(StringType())# Cast to String
            ,trim_chars_col         # Trim spaces, commas, and quote marks
           )
         .alias("Session Property Name") # Rename column
        
        # Second column selected
        ,functions.trim(
            functions.split(
                value_col
               ,colon_col
             )[1]                    # Second token from split
               .cast(StringType())
            ,trim_chars_col
           )
         .alias("Session Property Value")
     )    
    )
    return session_info_DF
# end get_session_info_DF(Session)  

def display_session_info(sess:Session) -> None:
    get_session_info_DF(sess).show()
    
def print_session_info(sess:Session) -> None:
    print(str(sess._session_info))


---

<a id="os_info"></a>

#### OS Information

In [7]:
def get_os_env_variable(variable_name:str) -> str:
    import os
    return os.environ[variable_name]

def get_jupyter_hub_user_from_OS() -> str:
    return get_jupyter_hub_username_from_OS()

def get_jupyter_hub_user_name_from_OS() -> str:
    return get_jupyter_hub_username_from_OS()

def get_jupyter_hub_username_from_OS() -> str:
    return get_os_env_variable("JUPYTERHUB_USER")

def get_snowflake_account_from_OS() -> str:
    return get_jupyter_hub_user_from_OS().split("-")[0]

def get_snowflake_user_from_OS() -> str:
    return get_snowflake_username_from_OS()

def get_snowflake_user_name_from_OS() -> str:
    return get_snowflake_username_from_OS()

def get_snowflake_username_from_OS() -> str:
    return get_jupyter_hub_user_from_OS().split("-")[1]

def get_python_version() -> str:
    from platform import python_version
    return python_version()

def python_version() -> str:
    return get_python_version()

def get_python_version_tuple() -> tuple:
    import sys
    return (sys.version_info.major,sys.version_info.minor,sys.version_info.micro)

def python_version_tuple() -> tuple:
    return get_python_version_tuple()

def python_major_version() -> int:
    return python_version_tuple()[0]

def python_minor_version() -> int:
    return python_version_tuple()[1]

def python_micro_version() -> int:
    return python_version_tuple()[2]

def get_python_major_version() -> int:
    return python_major_version()

def get_python_minor_version() -> int:
    return python_minor_version()

def get_python_micro_version() -> int:
    return python_micro_version()

---

<a id="version_checking"></a>
#### Version Checking 

In [8]:
def using_snowpark_major_release_version(version:int) -> bool:
    return snowpark_major_release_version() == version

def using_snowpark_minor_release_version(version:int) -> bool:
    return snowpark_minor_release_version() == version

def using_snowpark_maintenance_release_version(version:int) -> bool:
    return snowpark_maintenance_release_version() == version

def using_python_version(version:str) -> bool:
    return get_python_version() == version

def using_python_major_version(version:int) -> bool:
    return python_major_version() == version

def using_python_minor_version(version:int) -> bool:
    return python_minor_version() == version

def using_python_micro_version(version:int) -> bool:
    return python_micro_version() == version

---

<a id="packages"></a>
#### Available Packages

Function `packages_dataframe` returns data from `INFORMATION_SCHEMA.PACKAGES`. By default, packages from all languages (Python, Scala, and Java) are included. Booleans can be passed to narrow the results. And a `count` can be customized as well. 

In [9]:
def packages_dataframe(sess,python=True,java=True,scala=True,count=10) -> DataFrame:    
    sql_text = "SELECT * FROM INFORMATION_SCHEMA.PACKAGES WHERE "
    if(python):
        sql_text = sql_text + "LANGUAGE ILIKE 'python'"
        if(java):
            sql_text = sql_text + " OR "            
        
    if(java):
        sql_text = sql_text + "LANGUAGE ILIKE 'java'"
        if(scala):
            sql_text = sql_text + " OR "
        
    if(scala):
        sql_text = sql_text + "LANGUAGE ILIKE 'scala'"
        
    if((python==False) & (java==False) & (scala==False)):
        sql_text = sql_text + "FALSE"
    
    return sess.sql(sql_text)

def get_packages_dataframe(sess,python=True,java=True,scala=True,count=10) -> DataFrame:
    return packages_dataframe(sess,python=python,java=java,scala=scala,count=count)

def show_scala_packages(sess,count=10) -> None:
    packages_dataframe(sess,python=False,java=False,scala=True,count=count).show() 
    
def show_python_packages(sess,count=10) -> None:
    packages_dataframe(sess,python=True,java=False,scala=False,count=count).show()
    
def show_java_packages(sess,count=10) -> None:
    packages_dataframe(sess,python=False,java=True,scala=False,count=count).show()
    
def show_java_and_scala_packages(sess,count=10) -> None:
    packages_dataframe(sess,python=False,java=True,scala=True,count=count).show()       

---

<a id="clean"></a>

#### Close the Session and Clean up the Context Items


In [10]:
# Drops the passed schema if it exists and suspends the warehouse 
def close_session_and_clean_up(lesson:Lesson) -> None:
    if(lesson == None):
        print("Lesson object (lesson) not found!")
        print("Perhaps function confirm_or_create_lesson_context(...) was never invoked at the start of this notebook.")
        return None
    print("Lesson object (lesson) found.")
    if(lesson.get_session() == None):
        print("No snowflake.snowpark.Session object in scope to close.")
        return None
    if(lesson.is_session_closed() == True):
        print("Session already closed.")
        return None
    else:                
        try:
            print(f"Dropping schema {lesson.full_schema()}")
            (lesson.get_session().sql(f"DROP SCHEMA IF EXISTS {lesson.full_schema()}")
                .show(1,80)
            )
        except Exception as ex:    
            print("Trouble dropping schema. Session might have already been closed.")
            print(f"Message: {str(ex)}")
            
    if (lesson.contextDF == None):
        print(f"Context items weren't created. Can't suspend any warehouse.")
        return None
    
    try:
        print(f"Suspending warehouse {lesson.warehouse()}")
        (lesson.get_session().sql(f"ALTER WAREHOUSE {lesson.warehouse()} SUSPEND")
            .show(1,80)
        )
    except Exception as ex:    
        print("Trouble suspending warehouse. It was likely not running or session was closed")
        print(f"Message: {str(ex)}")
        
    try:
        print("Closing session")
        lesson.close_session()
        print("Session closed")
    except Exception as ex:    
        print("Trouble closing session")
        print(f"Message: {str(ex)}")
# end close_session_and_clean_up(Lesson)        